This notebook helps you learning about the openCV library (https://opencv.org).
API documentation is at the link https://docs.opencv.org/4.7.0/ (you may need to change the library version) 

Documentation refers to the main modules of the library:

- core: Core functionality
- imgproc: Image Processing
- imgcodecs: Image file reading and writing
- videoio: Video I/O
- highgui: High-level GUI
- video: Video Analysis
- calib3d: Camera Calibration and 3D Reconstruction
- features2d: 2D Features Framework
- objdetect: Object Detection
- dnn: Deep Neural Network module
- ml: Machine Learning
- flann: Clustering and Search in Multi-Dimensional Spaces
- photo: Computational Photography
- stitching: Images stitching
- gapi: Graph API

Documentation is mainly in C++ but wrapper functions for Python are indicated as well. 


The library can be installed in your Python environment by using "pip install opencv-python"
The command will install an unofficial pre-built OpenCV package for Python

Once the library is installed, you can import it

openCV 2 is written in C++ and uses several libraries, especially to handle image formats and video codec. 
Some functionalities may not work if the pre-built openCV library has been compiled with external libraries that are actually missing in your machine.

When this occurs, you need to check the openCV's Build information and check if you are missing some of them!! In general, issues are solved by installing the missing libraries.

You need to check specifically the libraries for: GUI (since windows will not work if there are issues), Video I/O (since you will have troubles opening videos).

In [ ]:
import cv2

print(cv2.getBuildInformation()) #this command will print the building information

Images are treated as N-channels numpy array of 8 bit unsigned int.
Color images are made of 3 channels and openCV stores them as BGR  (not RGB). This is important to remember when you wanna process them.

To deal with the images, we will also import numpy

In [ ]:
import numpy as np

We will start by reading an image.

Images can be stored in different formats, depending also on the compression level used to store the image.

Common formats are (many other exist!):
- Windows bitmaps – *.bmp, *.dib
- JPEG files – *.jpeg, *.jpg
- Portable Network Graphics – *.png 
- TIFF files – *.tiff, *.tif 

To load the image from the HDD, we use the method imread() which is able to handle all above formats.
The loaded image will be a C-dim numpy array of size HxWxC where H = height, W=width, C=Channels
You need to think it as a 2D pixel arrays for C color channels.

We will print the size of the image using the numpy method shape, which returns (H,W,C)

We will access each pixel by a pair of indices [row, col]

In [ ]:
im = cv2.imread("Lenna.png")

print("data type: ", type(im))
print("image size: ", im.shape)
print("element data type: ", im.dtype)

print("First pixel, blue channel: ", im[0,0][0])
print("Last pixel in first row, red channel: ", im[0,im.shape[1]-1,2])

print("First pixel in last row, green channel: ", im[im.shape[0]-1,0,1])
print("Last pixel in last row, red channel: ", im[im.shape[0]-1,im.shape[1]-1,2])


We can access an entire row or coloumn of any channel using ":" or defining index ranges
We can also define a sub-image (RoI).

Don't get confused with the order by which numpy array are printed in python!! Parenteses are the key to read properly the data!


In [ ]:
print(im[0,:, 0]) # first row of the image - blue channel
print("green ch:")
print(im[0:3, 0:3, 1]) #3x3 sub image - green channel

sub_im = im[0:3,0:3] #3-channel subimage
green_ch  = sub_im[:,:,1]
print("sub-image:")
print(sub_im)
print("green ch:")
print(green_ch)

We can also visualize an image by using the openCV method imshow().

GUI in opencv has methods for creating windows, destroying windows.
Windows are characterized by a name, which is a string used as ID for the window.

cv2.namedWindow(window_name, flag)  
cv2.destroyWindow(window_name)
cv2.destroyAllWindows()

Documentation for the meaning of flag: https://docs.opencv.org/4.7.0/d7/dfc/group__highgui.html#ga5afdf8410934fd099df85c75b2e0888b

The method imshow will create a window and display the image. However, we need to give time to the system for the rendering. Hence, we need to add a pause. This can be accomplished by using the method cv2.waitKey(delay).

delay = 0 means that it will pause until a key is not pressed (focus on the window first)
delay is expressed in ms

Check the documentation for more details!

Note: be aware that within the notebook, destroyAllWindows may not work. Try it in a separate project.

In [ ]:
sub_im = im[250:, 200:]
cv2.imshow("Lenna", im)
cv2.imshow("Lenna_sub", sub_im)
cv2.waitKey(0)
cv2.destroyAllWindows()

Here you need to be very careful!!

When we define RoI (sub-images) we are not creating a copy but a reference to a portion of the image

This is also true when you assign an image to another variable (this is how numpy works!)

Let's inspect the following code.

In [ ]:
sub_im = im[250:, 200:]
sub_im[:,:,0:2] = 0 # blue and green channels are put to 0 values. This change affects also im

cv2.imshow("Lenna", im)
cv2.imshow("Lenna_sub", sub_im)
cv2.waitKey(0)
#cv2.destroyAllWindows()

Putting to 0 the blue and the green channels in sub_im has the effect that we only see the red channel.
Since sub_im is a refence, we have changed also the image im.

the following code will assign the image to another variable. Changing the values in this variable will have effects in the original image and in the RoI

In [ ]:
im_copy_ref = im

im_copy_ref[250:, 200:, 2] = 0

cv2.imshow("Lenna", im)
cv2.imshow("Lenna_2", im_copy_ref)
cv2.imshow("Lenna_sub", sub_im)
cv2.waitKey(0)
#cv2.destroyAllWindows()

Please, be very careful when you handle references!

If we need to make changes to the sub_image or to a copy of the image without affecting the original one, you need to make a deep copy!



In [ ]:
im = cv2.imread("Lenna.png")
im_deep_copy = im.copy()

im_deep_copy[250:, 200:, :] = 0
cv2.imshow("Lenna", im)
cv2.imshow("Lenna_2", im_deep_copy)
cv2.waitKey(0)


If we want to save an image, we need to use the method cv2.imwrite(	filename, img[, params]	).

The method has a lot of parameters (whose meaning is clear when you know about the image format) that you should check in the documentation. You can also use default values. The method returns a boolean indicating the success of the operation.

In [ ]:
ret = cv2.imwrite("new_lenna.png", im_deep_copy)
print(ret)

We can also use the numpy library to create new images. To be visualized or stored, image values must be in the range [0 255] and the data type must be uint8.

If instead we wanna manipulate images, we need to be careful to the data type! In fact, it is better to cast the data type to float or double before manipulating the image and the casy back to the uint8 format.
Sometimes, we need to renormalize the data in the range [0 255].

In [ ]:
black_im = np.zeros((512, 256), np.uint8)
white_im = 255*np.ones((512, 256), np.uint8)

cv2.imshow("black", black_im)
cv2.imshow("white", white_im)
cv2.waitKey(0)


Let's say we want to compute the average between two images. This is defined as the average of the intensities  pixel-by-pixel. While in C-language we would use 2 for cycles, numpy allows to compute the average image easily.
Here we need to check: i) the shapes of the matrices, which must be the same; ii) the data type; iii) the range of the output matrix. 

In [ ]:
gray_im = 0.5*(black_im.astype(np.float32) + white_im.astype(np.float32) )
gray_im = gray_im.astype(np.uint8) #without the cast, data are interpreted in the wrong way...
cv2.imshow("gray", gray_im)
cv2.waitKey(0)

new_image = np.random.randint(0, 1000, (256,256,3)).astype(np.uint8) #It crops!
print("max: ", np.max(new_image), " min: ", np.min(new_image))

#The correct way is:
new_image = np.random.randint(0, 1000, (256,256,3))
print("max: ", np.max(new_image), " min: ", np.min(new_image), "data type: ", new_image.dtype)
new_image = cv2.normalize(new_image, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)
print("max: ", np.max(new_image), " min: ", np.min(new_image), "data type: ", new_image.dtype)

cv2.imshow("rand", new_image)
cv2.waitKey(0)


One common operation we may need to do is rescaling an image.  This can be done with the method resize().
We can also write an image on a specific path.



In [ ]:
import numpy as np
import cv2

image = cv2.imread("Lenna.png")
image2 = cv2.resize(image.copy(), (256, 256))
image3 = cv2.resize(image.copy(), (0,0), fx=0.5, fy=0.5)

print("original: ", image.shape)
print("image2: ", image2.shape)
print("image3: ", image3.shape)

cv2.imshow("original", image)
cv2.imshow("256x256", image2)
cv2.imshow("Halved image", image3)
cv2.waitKey(0)